# GenAI System Architecture Overview

A practical guide to designing, scaling, and documenting production-grade GenAI systems — built around a running example of **GloboAssist**, a real-time customer support platform for a mid-size e-commerce company.

## Table of Contents

1. [Module 01: Understanding Core GenAI System Components](#module-01-understanding-core-genai-system-components)
2. [Module 02: Evaluating Architectural Patterns and Trade-offs](#module-02-evaluating-architectural-patterns-and-trade-offs)
3. [Module 03: Scaling and Optimizing GenAI Performance](#module-03-scaling-and-optimizing-genai-performance)
4. [Module 04: Integrating and Designing APIs for GenAI Systems](#module-04-integrating-and-designing-apis-for-genai-systems)
5. [Module 05: Documenting Architecture Decisions](#module-05-documenting-architecture-decisions)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

## Module 01: Understanding Core GenAI System Components

A production GenAI system is not just a model — it's a full stack. This module introduces the four-layer architecture, how the layers interact through clean contracts, and the key architectural decisions (sync vs. async, stateful vs. stateless, monolith vs. microservices) that shape every system before a line of code is written.

### The Running Example: GloboAssist

**Globomantics** is a mid-size e-commerce company with 50,000+ products, a NA + EU customer base, and 2,000 support tickets per month. Their current support process averages **12 minutes per ticket** — manually searching the knowledge base, copy-pasting from FAQ pages, and typing replies from scratch.

**GloboAssist** is the GenAI platform they are building to address this. It has three distinct workloads on one platform:

| Workload | Type | Description |
|---|---|---|
| Customer-facing chat | Real-time | Customer support conversations with product Q&A |
| Internal agent copilot | Real-time | Suggestions and reply drafts for support agents |
| Batch analytics pipeline | Asynchronous | Ticket mining, trends, and weekly summaries |

The course follows the evolution of GloboAssist from a minimal MVP (a thin Python LLM wrapper) to a production-grade architecture, making every architectural decision along the way.

**Assumed knowledge:** LLM concepts and distributed systems fundamentals. **Focus:** GenAI-specific architectural decisions.

### The Anatomy of a Production GenAI System

#### Why a Thin LLM Wrapper Fails

The simplest possible GenAI system is a Python service that wraps an LLM API directly — no retrieval, no memory, no guardrails. Reality hits fast:

- **Hallucinated specs** — the model claims IP67 when the product is IPX4
- **No memory across turns** — every turn starts from scratch ("As I was saying…" goes nowhere)
- **No access to live data** — "Where is my order?" sends the customer back to the website
- **Generic, off-brand voice** — sounds nothing like the company

> A production GenAI system isn't just a model. It's a full stack.

#### The Four-Layer Architecture

| Layer | Role | Solves |
|---|---|---|
| **Model** | The brain — LLM for text generation, embeddings, classification | Core language understanding |
| **Data** | The knowledge — vector DB, product catalog, order history, session store | Hallucination and grounding |
| **Orchestration** | The coordinator — intent classification, context assembly, guardrails, multi-step workflows | Routing and domain logic |
| **Serving infrastructure** | The front door — API gateway, load balancer, auth, rate limiting, observability | Reliability and access control |

Remove any one layer and the system breaks: the model can't answer without data, the data is useless without orchestration, and none of it reaches users without serving.

In [ ]:
render_diagram(Path("assets/01-orchestration-layer-flow.png"))

### How the Layers Interact: Contracts

Knowing the layers exist is not enough — what matters is how they talk to each other. Poor interfaces turn layers into a monolith in disguise: clean on the whiteboard, fragile in practice.

#### Serving → Orchestration

The API layer passes to the orchestrator:
- **Request:** user message, session ID, metadata
- **Response:** generated text, citations, confidence scores

#### Orchestration → Data

The orchestrator does not need to know where data is stored. It decides *when* to retrieve, *what* to retrieve, and *how much* context to inject. The data layer exposes three contract types:
- **Vector database:** embedding + filters → ranked chunks
- **SQL database:** query → order records
- **External API:** call → CRM data

#### Orchestration → Model

A complete model call includes:

```json
{
  "model": "claude-sonnet-4-6",
  "messages": [
    { "role": "system", "content": "You are a support assistant…" },
    { "role": "user",   "content": "Does the X1 support USB-C?" },
    { "role": "system", "content": "<context>X1 specs…</context>" }
  ],
  "temperature": 0.2,
  "tools": ["lookup_order"]
}
```

Four things GloboAssist always sends: (1) system prompt establishing the assistant role, (2) injected context from the data layer, (3) low temperature for factual/deterministic answers, (4) tools array so the model can look up order data.

**The trade-off of layering:**

| Gains | Costs |
|---|---|
| Scale, test, deploy each side independently | Network latency between layers |
| Swap your vector DB without touching orchestration | Added operational complexity |
| Change model providers without rewriting serving | More moving parts to monitor |

> Contracts are what make the architecture modular, not just layered on a diagram.

### Tracing Requests from Input to Output

A static diagram doesn't answer: what fires first, what can run in parallel, and where does latency accumulate? The three GloboAssist workloads show how the same four layers support radically different execution paths.

In [ ]:
render_diagram(Path("assets/02-customer-chat-sequence.png"))

In [ ]:
render_diagram(Path("assets/03-agent-copilot-sequence.png"))

In [ ]:
render_diagram(Path("assets/04-analytics-batch-sequence.png"))

**Same four layers, three very different optimization targets:**

| Workload | Latency budget | Bottleneck | Serving pattern | Optimized for |
|---|---|---|---|---|
| Customer chat | < 2 s | LLM inference | SSE streaming | Speed |
| Agent copilot | < 3 s | Retrieval fan-out | REST response | Depth |
| Analytics | < 30 min | Throughput + cost | Job queue | Cost |

> Architectural decisions aren't one-size-fits-all.

### Key Architecture Decision Points

Three decisions that are expensive to change later — make them before you build.

#### 1. Synchronous vs. Asynchronous

| Mode | When to use | Trade-off |
|---|---|---|
| **Synchronous** | A person is waiting for a real-time response | Simpler to build, but in GenAI you're blocking on operations that take seconds, not milliseconds |
| **Asynchronous** | Batch jobs; sync would time out | Better throughput, but pays in real complexity (message queues, polling, error recovery) |

GloboAssist mapping: Customer chat → Sync, Agent copilot → Sync, Analytics → Async.

#### 2. Stateless vs. Stateful

| Mode | Scaling characteristic | Risk |
|---|---|---|
| **Stateless** | Scales horizontally — any instance handles any request | Where does conversation memory live? |
| **Stateful (sticky sessions)** | Can hold history in memory | Instance crash = conversation gone |

**Resolution:** Keep the orchestrator stateless. Store session state in an external fast key-value store (Redis). This gives horizontal scaling without losing multi-turn context.

In [ ]:
render_diagram(Path("assets/05-stateless-stateful.png"))

#### 3. Monolith vs. Microservices vs. Something In Between

| Approach | Pro | Con |
|---|---|---|
| **Monolith** | Fast to build, easy to debug | Scaling bottlenecks are coupled |
| **Microservices** | Independent scaling | Network and ops overhead |
| **Modular monolith** | Clear module boundaries | Split only when scaling profiles diverge |

**GloboAssist decision:** Start as a modular monolith. Split out only two services that have distinct scaling profiles: the **retrieval service** (scales with query volume) and the **analytics workers** (scale with ticket backlog).

#### Decision Matrix Summary

| Decision | Choice |
|---|---|
| Sync vs. async | Sync for chat + copilot, async for analytics |
| Stateful vs. stateless | Stateless orchestrator + Redis session store |
| Monolith vs. microservices | Modular monolith; split retrieval and analytics workers |

These are spectrums, not binary choices. Chat is sync, agent is sync, batch is async. The model layer is stateless; the data layer holds state. Make the call per workload and per layer.

## Module 02: Evaluating Architectural Patterns and Trade-offs

Your use case must dictate your architecture, not the other way around. This module covers the four core GenAI architectural patterns (RAG, agents, fine-tuning, hybrid), a systematic approach to evaluating trade-offs, and the implications of hosting strategy choices.

### Demo: Architectural Trade-off Evaluation

See [`02-tradeoff-evaluation-demos/`](02-tradeoff-evaluation-demos/) for the weighted decision matrix spreadsheet used to evaluate vector database options for GloboAssist.

### The Four Core Patterns

#### RAG — Retrieval-Augmented Generation

Retrieve relevant documents, inject them into the prompt, then generate.

| Strengths | Weaknesses |
|---|---|
| Grounded in real data | Retrieval quality is a bottleneck |
| No retraining when info changes | Limited by context window size |
| Knowledge base stays current | Retrieval step adds latency |

**GloboAssist fit:** Customer chat. The product knowledge base updates frequently; answers must reference the latest product data, not what the model memorized during training.

In [ ]:
render_diagram(Path("assets/06-rag-flow.png"))

#### Agent-Based Systems

The LLM decides which tools to call, in what order, and with what parameters.

| Strengths | Weaknesses |
|---|---|
| Flexible and extensible — add tools without retraining | Behavior harder to predict |
| Handles multi-step workflows | Higher latency — multiple LLM calls |
| | Errors propagate across steps |

**GloboAssist fit:** Internal agent copilot. Multi-step troubleshooting needs tool orchestration; the sequence of calls depends on what each previous call returns (e.g., check SSO config → query identity provider → search recent tickets → look up account in CRM).

In [ ]:
render_diagram(Path("assets/07-agent-copilot-flow.png"))

#### Fine-Tuning

Customize the model's weights on your domain-specific data.

| Strengths | Weaknesses |
|---|---|
| Faster inference — no retrieval step | Training is expensive |
| Better on specialized tasks | Needs a robust data pipeline |
| Less context needed per prompt | Risk of model drift; slower to update |

**GloboAssist fit:** Tone and voice. Stylistic consistency is hard to achieve with prompting alone. Fine-tuning makes the model sound like the brand:

| Approach | Example response |
|---|---|
| Generic (untuned) | "I understand your concern. Please provide your order number so I can investigate this matter for you." |
| Fine-tuned (on-brand) | "Hey, sorry you're hitting this. Drop your order # and I'll jump in and sort it out right now." |

#### Hybrid Architectures

Most production systems combine patterns:
- **Fine-tuned model + RAG** — domain voice with up-to-date knowledge
- **Agent + RAG** — grounded facts in multi-step reasoning

**GloboAssist is a hybrid platform:**

| Workload | Pattern |
|---|---|
| Customer chat | RAG |
| Agent copilot | Agent + RAG + fine-tuning |
| Analytics | Batch RAG |

#### Pattern Comparison

| Pattern | Data freshness | Latency | Cost | Complexity |
|---|---|---|---|---|
| RAG | Highest | Medium | Low | Medium |
| Agents | High | High | High | Highest |
| Fine-tuning | Low | Low | High | Medium |
| Hybrid | High | Varies | High | Highest |

> Choose the right pattern for each workload, not one pattern for every workload.

### Evaluating Trade-offs: The Triangle

Optimize one dimension and you pay in the others. There is no perfect architecture — only choices.

In [ ]:
render_diagram(Path("assets/08-trade-off-triangle.png"))

Every decision is a concrete trade-off across the three vertices:

| Decision | Latency | Accuracy | Cost |
|---|---|---|---|
| Retrieve 15 chunks instead of 5 | ↓ Slower | ↑ Better | ↓ Pricier |
| Switch to a smaller, faster model | ↑ Faster | — Risk | ↑ Cheaper |
| Add a semantic cache | ↑ Faster | — Stale risk | ↑ Cheaper |

**How to use the triangle — four steps:**

1. **Define thresholds** — minimum acceptable per dimension
2. **Pick a primary dimension** — the one that matters most for this workload
3. **Optimize it** — while keeping the others above their thresholds
4. **Measure and iterate** — re-check as things evolve

**GloboAssist primary dimensions:**
- Customer chat → Latency (customers won't wait)
- Agent copilot → Accuracy (bad suggestions destroy trust)
- Analytics → Cost (must fit the budget)

> Define thresholds, pick a primary dimension, evaluate against both. Frameworks outlast models — models and prices will change, but this discipline won't.

### Model Hosting Strategies

Your hosting choice reshapes the entire architecture — affecting latency, cost, and legal compliance.

#### Three Strategies

| Strategy | Examples | Benefits | Trade-offs |
|---|---|---|---|
| **Third-party APIs** | OpenAI, Google | Fast to start; no infra; pay per token | Data leaves your network; retry/fallback/rate-limit handling required |
| **Self-hosted open source** | Gemma, Mistral on your GPUs | Full data control; better unit economics at scale | GPU provisioning, VRAM planning, versioning, health monitoring |
| **Hybrid** | Mix of both | Different hosting for different workloads | Most operational complexity |

#### VRAM Planning for Self-Hosted Models

A critical but often underestimated cost. Weights alone are just the starting point — KV cache, activations, and runtime overhead add more on top:

| Model size | FP16 weights | Single GPU (24 GB) | With 4-bit quantization |
|---|---|---|---|
| 7B params | ~14 GB | Fits on one 24 GB GPU | ~7 GB — fits easily |
| 70B params | ~140 GB | Requires 2× 80 GB or 4× 40 GB GPUs | ~35–40 GB → fits on a single 80 GB GPU |

Quantization degrades quality minimally and is often the right call for 70B+ models.

#### Compliance Drives Architecture

It's not just a technical decision — regulations can rule out hosting options regardless of cost or performance. Example: GDPR requires that EU customer data (name, order ID, address) stays within EU-hosted infrastructure. This means requests containing PII must be routed only to EU-hosted models.

Also verify:
- Whether customer data is used for model training (read the DPA carefully)
- Industry regulations: HIPAA, SOC 2, PCI-DSS may further constrain options

In [ ]:
render_diagram(Path("assets/09-hosting-architecture.png"))

### A Systematic Approach to Architectural Decisions

Two traps teams fall into: decision paralysis (analyzing for months, never committing) and gut-feel choices (a decision you can't justify to leadership). A repeatable four-step framework solves both.

#### Step 1: Define Constraints and Requirements

- **Hard constraints** — eliminate options outright: regulatory compliance, budget ceilings, existing infrastructure
- **Soft preferences** — nice to have: team familiarity, preference for managed services, vendor relationships

**GloboAssist example:**

| Hard constraints | Soft preferences |
|---|---|
| 50,000 products | PostgreSQL expertise |
| Retrieval latency < 200 ms | Prefers managed services |
| Budget < $500/month | |
| GDPR: EU data stays in EU | |

#### Step 2: Evaluate Alternatives with a Weighted Decision Matrix

- Always consider at least three options (two is binary thinking and usually misses a better middle path)
- Assign weights reflecting business priorities, not just technical merit
- Score each option 1–5 against every criterion

**Example: Vector database selection**

| Criterion | Weight | Pinecone | Qdrant | pgvector |
|---|---|---|---|---|
| Compliance | 4 | 3 | 3 | 5 |
| Performance | 4 | 5 | 4 | 3 |
| Cost | 3 | 2 | 3 | 5 |
| Scalability | 3 | 5 | 4 | 3 |
| Operational burden | 2 | 5 | 2 | 3 |
| Vendor lock-in | 2 | 2 | 3 | 5 |
| Time-to-implement | 1 | 5 | 3 | 4 |
| **Total** | | **72** | **62** | **76** |

Important: when weights shift, winners shift. With EU customers (compliance weight = 4) → pgvector wins. Without EU customers (compliance weight = 1) → Pinecone wins.

#### Step 3: Assess Trade-offs, Risks, and Long-Term Implications

Ask:
- What are you gaining?
- What risks does this introduce?
- What are you giving up?
- What dependencies does it create?
- Will this choice constrain future options? What's the migration cost?

**Choosing pgvector example:**

| Dimension | Assessment |
|---|---|
| Gain | Zero incremental cost and full compliance control |
| Give up | Best-in-class vector search performance at extreme scale |
| Risk | Migration needed if catalog grows past 200,000 products |
| Long-term | Postgres has a huge ecosystem and is very stable |

#### Step 4: Align with Stakeholders — Frame the Same Decision for Each Audience

| Audience | Their framing |
|---|---|
| Engineering | "We're leveraging our existing Postgres skills — no new infra to learn" |
| Finance | "$0 incremental cost vs. ~$500/month for the alternative" |
| Legal | "Full data residency control inside our EU datacenter" |
| Product | "3 days to implement instead of 2 weeks" |

Your CFO doesn't care about HNSW indexes. Legal doesn't care about millisecond latency. Frame the same decision differently for each audience.

## Module 03: Scaling and Optimizing GenAI Performance

In GenAI, you can't just add more servers to scale. Each layer in the architecture has fundamentally different scaling characteristics, and ignoring that leads to mismatched optimization and wasted spend.

### Demo: Bottleneck Identification

See [`03-performance-demos/`](03-performance-demos/) for a runnable Python demo that simulates retrieval, prompt construction, and model inference phases to visualize where latency accumulates. Run with `uv run bottleneck_demo.py` or `pip install -r requirements.txt && python bottleneck_demo.py`.

### Independent Scaling by Layer

Every layer has its own bottleneck type — match each layer's scaling strategy to its actual constraint:

| Layer | Bottleneck type | Scaling playbook |
|---|---|---|
| **Data** | I/O-bound (disk reads, index lookups) | Read replicas on vector DB; shard by collection; independent embedding service |
| **Model** | Compute-bound (GPU cycles per token) | Self-hosted: instances behind load balancer; third-party APIs: manage rate limits and connection pools |
| **Orchestration** | Memory-bound (session state, in-flight requests) | Stateless instances; auto-scale on queue depth or request rate |

**Two ways to scale anything:**
- **Vertical** — bigger CPU, RAM, or GPU on one instance. Great for vector DB and model inference. Limited by hardware ceiling.
- **Horizontal** — more instances behind a load balancer. Perfect for stateless components. Hard once you have state or consistency requirements.

In [ ]:
render_diagram(Path("assets/10-scaled-architecture.png"))

Customer chat and analytics are isolated paths to separate bottlenecks, separate SLAs, and separate model endpoints. Neither workload can accidentally affect the other.

### Identifying and Resolving Common Bottlenecks

#### Bottleneck 1: Slow Retrieval

**Symptoms:** High p95 on retrieval queries; retrieval > 30% of total latency

**Causes:** Unoptimized vector index; retrieving too many chunks; flat search without metadata pre-filtering

**Fixes:**
- Tune HNSW index parameters
- Add metadata pre-filtering
- Reduce `top_k`
- Cache frequent query embeddings

#### Bottleneck 2: Model Inference Latency

The request timeline has two distinct phases — TTFT (time to first token) and total generation. Each has different causes:

**Symptoms:** TTFT takes several seconds; total generation takes several minutes

**Cause:** Model size × prompt length — longer prompts mean more tokens to process before generation even starts

**Fixes:**
1. Route simple queries to a smaller model
2. Remove redundant context to compress prompts
3. Stream output so users see tokens as they appear
4. Batch requests when possible

#### Bottleneck 3: Context Window Limits

**Symptoms:** Responses truncated; model loses track of earlier conversation

**Cause:** Too much history and too many documents stuffed into a single prompt

| Approach | Mechanism | Trade-off |
|---|---|---|
| Bigger context window (tempting fix) | Just increase the limit | Higher latency and cost with no quality gain |
| Compress conversation | Summarize old turns with a cheap model | Preserves key facts; loses conversational nuance |
| Smart context selection | Keep only the most relevant retrieved chunks | Most accurate; hardest to build |

#### Bottleneck 4: Network Overhead

**Symptom:** Random latency spikes uncorrelated with traffic

**Fixes:**
- Sequential API calls → Parallel calls
- No connection reuse → Connection pooling
- Cross-region hops → Co-locate services

#### Bottleneck 5: Architectural Mismatch

Sometimes it's not a slow component — it's the wrong architecture. Trying to handle batch analytics synchronously will time out; the right fix is an async message broker with background workers, not tuning the synchronous path.

### Designing for Concurrent User Loads

Scaling alone doesn't solve concurrency. On launch day: 500 customers in chat simultaneously, 50 support agents on the copilot — hundreds of overlapping requests competing for the same connections and model endpoints. Without concurrency design: dropped requests, degraded experience.

#### Strategy 1: Connection Pooling

Per-request TCP handshake + TLS negotiation + authentication is invisible at 5 users, but saturates at 500. A reusable pool lets requests borrow a connection and return it when done.

**Sizing rule:** `concurrency × avg_request_duration`

#### Strategy 2: Request Queuing

Even with pooling, load sometimes exceeds capacity. Queue requests instead of dropping them.

Use a priority queue — higher-priority tiers are pulled first:

| Priority | Source |
|---|---|
| HIGH | Checkout flow; paying tier |
| MID | Logged-in user |
| LOW | Anonymous user |

Implementation notes:
- **Backing store:** Redis sorted set
- **Score:** tier + enqueue timestamp
- **Max depth:** ~200 before admit gates kick in
- **Timeout:** 30 s → return a graceful response, never hang the customer

#### Strategy 3: Multi-Level Caching

| Cache type | Mechanism | Example |
|---|---|---|
| **Response cache** | Exact string match | "What are your store hours?" → cached |
| **Semantic cache** | Cosine similarity ≥ 0.95 | "What's your return policy?", "Can I send this back?" → same cached answer |
| **Embedding cache** | Skip recomputation for unchanged content | Product catalog embeddings — don't re-embed on every request |

#### Strategy 4: Load Balancing Across Model Instances

- **Third-party APIs:** distribute across multiple keys or providers
- **Self-hosted:** distribute inference across GPUs using least-connections or weighted round-robin

In [ ]:
render_diagram(Path("assets/11-launch-day-concurrency.png"))

On launch day with 40% cache hit rate: 500 × 1.2 s ≈ 417 req/s → 40% served instantly from cache → only ~250 reach the LLM. Agent queries bypass the cache entirely (too context-specific).

### Measuring Performance: KPIs and Architecture

Metrics are where trade-offs play out in production. Use three tiers:

#### Tier 1: User-Facing Metrics

Averages hide the pain — avg 500 ms can coexist with p99 of 8 s, meaning 1 in 100 users has a terrible experience. Always track percentiles:

- p50, p95, p99 latency
- TTFT (time to first token)
- Error rate
- Availability / SLA metrics

#### Tier 2: System Health (predict problems before users feel them)

Watch for early warning signals:
- Throughput
- CPU / GPU utilization
- Memory
- Queue depth
- **Cache hit rate** — a drop from 40% → 10% means load hits the LLM directly, p99 spikes, and users feel it

#### Tier 3: GenAI-Specific Metrics (don't exist in traditional systems)

| Metric | What it measures |
|---|---|
| Tokens per request | Context window utilization % |
| Cost per query | Budget tie-back |
| Retrieval relevance score | RAG quality |
| Guardrail trigger rate | Prompt health |

**Healthy metrics are workload-specific:**

| Workload | Tier focus |
|---|---|
| Customer chat (real-time) | Latency percentiles, error rate, TTFT |
| Analytics (batch) | Cost per ticket, completion rate, job duration |

> p95 latency maps to model choice + data layer. Cost per query maps to token management. Cache hit rate maps to caching strategy. Metrics are where trade-offs play out in production.

## Module 04: Integrating and Designing APIs for GenAI Systems

GenAI APIs have unique requirements that traditional REST patterns don't address — in particular, the need to cancel a request while the model is still generating.

### Demo: API Integration Patterns

See [`04-api-integration-demos/`](04-api-integration-demos/) for two Python examples: `data_adapter.py` (the adapter pattern for swapping data sources) and `state_management_demo.py` (multi-turn conversation state with context window management). Run `state_management_demo.py` with `uv run state_management_demo.py`.

### GenAI-Specific API Requirements

#### Streaming First

GenAI APIs default to token streaming — the client receives an SSE event stream as the model generates tokens. Batch (non-streaming) mode is still available but should not be the default for user-facing workloads.

#### Cancellation Support

Without cancellation: users close chats or rephrase mid-sentence, the model keeps generating tokens nobody reads, and you pay for every one of them.

**Implementation:**
- Use request IDs with a dedicated cancel endpoint (`DELETE /v1/chat/{request_id}`)
- Handle WebSocket close events
- Propagate cancellation all the way to the LLM call

#### Structured Error Handling

Don't just return HTTP 500. Return typed, actionable error codes:

```
model_timeout | context_too_long | guardrail_blocked | rate_limited | retrieval_failure
```

#### Abstraction with Control

- **Hidden by default:** prompt templates, retrieval chunk counts, model selection (consumers don't need to understand the architecture)
- **Power user controls:** `temperature`, `max_tokens`, `retrieval_mode` (advanced integrators can tune without forking the backend)

#### Graceful Fallback

When the LLM is unavailable or the vector DB is unreachable, fall back to keyword search over the knowledge base rather than returning an error.

### GloboAssist's Three API Contracts

| Endpoint | Type | Purpose |
|---|---|---|
| `POST /v1/chat` | Streaming SSE | Customer-facing. Streams tokens in real time. Accepts `session_id` for multi-turn context. |
| `POST /v1/agent-assist` | Synchronous JSON | Internal agent console. Returns structured JSON: summary, suggested reply, confidence, sources. |
| `POST /v1/analytics/reports` | Async job polling | Batch weekly reports. Returns `job_id`; consumer polls `GET` endpoint for status and results. |

**Sample customer chat request/response:**

```json
// Request
POST /v1/chat
{
  "message":    "How do I reset…",
  "session_id": "sess_4f9a",
  "stream":     true,
  "options": { "max_tokens": 512 }
}

// SSE Response stream
event: token
data: {"token":"Resetting"}
event: token
data: {"token":" your"}
event: metadata
data: {"finish_reason":"stop"}
event: done
data: {}
```

Cancel at any time: `DELETE /v1/chat/{request_id}`  
Fallback: If LLM unavailable → `{"fallback": true, "results": [...]}` with keyword search results

### Enterprise Integration Patterns

#### Authentication

Don't build your own. Use OAuth 2.0 / OpenID Connect with an identity provider. The user context (role, permissions, scope) flows from the IdP to the orchestrator:
- Support agents see their own queue, not another team's
- Customers asking about orders see only their own orders

#### Database Integration

Two rules that prevent a class of security vulnerabilities:
1. **Read-only connections** — the orchestrator queries data, it does not modify it
2. **No LLM-generated SQL** — this is a massive injection risk. The LLM calls a data access layer that runs parameterized queries, it never constructs SQL strings directly

#### Logging and Compliance

Log every interaction as append-only, structured JSON with PII redacted:
- User input
- Model used
- Latency
- Retrieved chunks
- Model output

Use distributed tracing (with a shared `request_id`) so you can follow a single request from API gateway → orchestrator → vector DB → LLM → response.

#### Monitoring and Alerting

Emit to your existing observability stack (Prometheus/Grafana). Add GenAI-specific alert conditions on top of classic CPU/memory/latency alerts:
- Guardrail trigger rate > 5%
- Cost per query > threshold

### Modular Architecture and Separation of Concerns

Without a clear separation of concerns, a change to one component (e.g., swapping the embedding model) causes cascade failures across prompt templates and the orchestrator. The solution is to decompose the pipeline into three modules with explicit contracts.

In [ ]:
render_diagram(Path("assets/13-pipeline-decomposition.png"))

#### The Three Modules and Their Contracts

**Retrieval module**

Owns: embedding generation, vector search, reranking

```python
def retrieve(query: str, top_k: int, filters: dict) -> List[Document]:
    ...
```
Internally can use any vector DB — as long as the contract holds.

**Generation module**

Owns: prompt templates, model selection, LLM API calls

```python
def generate(messages: List[Message], context: List[Document], options: Options) -> Response:
    ...
```
Swap the model — nothing outside this module changes.

**Post-processing module**

Owns: guardrails, PII redaction, citation extraction, response formatting

```python
def process(raw_response: Response, policies: List[Policy]) -> SafeResponse:
    ...
```
Sanitized, safe, ready for the user.

#### The Adapter Pattern

The orchestrator depends only on the `DataAdapter` interface, not on any specific external system. New data sources implement the same interface — the orchestrator never needs to change.

In [ ]:
render_diagram(Path("assets/12-adapter-pattern.png"))

#### Contracts and Versioning

A contract defines: inputs, outputs, error types, and performance guarantees (e.g., p95 < 200 ms). Contracts are versioned — internals can change freely, but a schema change triggers a major version bump and requires testing before deploy.

**Three testing levels:**

1. **Unit tests** — no external dependencies, everything mocked
2. **Contract tests** — verify each module honors its published interface
3. **Integration tests** — run across the full pipeline (most expensive)

**The real payoff of modular contracts:**

| Benefit | Example |
|---|---|
| Independent deployment | Ship a new embedding model without touching generation |
| Easier debugging | p95 spike? Isolate it to one module — each has its own metrics |
| Team autonomy | Different teams, different cadences, different stacks |

> The contract is the boundary. The boundary is what makes it work.

### Managing State in Multi-Turn Conversations

Each turn in a conversation depends on context from earlier ones. The trap: sending the whole conversation history every turn causes prompts to grow unbounded, latency to spike, and costs to explode.

#### Session Storage

- **Don't:** store session state in the orchestrator's local memory — loses everything on instance restart
- **Do:** use a fast external key-value store like Redis with key `{user_id}:{session_id}` and TTL of 30 minutes of inactivity

#### Context Window Management

Your context window is a fixed token budget with three competing claims:

| Claim | Priority |
|---|---|
| System prompt | Non-negotiable |
| Retrieved KB context | Non-negotiable |
| Conversation history | Whatever's left |

**Truncation strategies:**

| Strategy | Mechanism | Trade-off |
|---|---|---|
| Sliding window | Keep only the last N messages | Easy to implement; forgets early context |
| Summarization | Cheap model condenses older turns | Preserves key facts; loses nuance |
| Selective retention | Extract and keep key entities only | Most accurate; hardest to build |

#### Session Recovery

Sessions get interrupted — network drops, browser refreshes, phone goes to sleep. The recovery flow:

1. User reconnects and sends `session_id`
2. Orchestrator rehydrates state from Redis
3. Conversation resumes where it left off

Purge guest conversations immediately at TTL expiry.

> Stateless architecture + external store + smart truncation = stateful user experience, without blowing the budget.

## Module 05: Documenting Architecture Decisions

Most architecture documentation falls into two traps: too abstract ("we use microservices") or too detailed (200 lines of YAML). Neither serves the people who need it.

### Demo: Architecture Documentation Artifacts

See [`05-architecture-docs-demos/`](05-architecture-docs-demos/) for complete working examples:
- `clip02/` — C4 context and container diagrams in Mermaid format
- `clip03/` — Sequence and data flow diagrams (including optimistic streaming)
- `clip04/` — A complete ADR: `ADR-003-pgvector.md`
- `clip05/` — Architecture roadmap, debt registry (CSV, YAML, Markdown), and stakeholder communication templates

Render Mermaid diagrams at [mermaid.live](https://mermaid.live) or with the Mermaid CLI.

### The C4 Model: Multi-Level Documentation

One diagram cannot serve everyone. Executives need scope; architects need structure; developers need operational detail; compliance needs data lineage. The C4 model solves this with four zoom levels — each audience stops at the right level.

#### Level 1: Context

Your system as a single black box. Shows external users and external systems it interacts with. Executives stop here.

In [ ]:
render_diagram(Path("assets/14-c4-level1-context.png"))

#### Level 2: Container

Zoom inside the system: applications and data stores. Architects stop here.

In [ ]:
render_diagram(Path("assets/15-c4-level2-container.png"))

#### Level 3: Component

Zoom inside one container: components and their responsibilities. Developers stop here.

In [ ]:
render_diagram(Path("assets/16-c4-level3-component.png"))

#### Level 4: Code

Zoom inside one component: classes and methods. Rarely used in practice — the code itself serves this level.

#### Write Diagrams as Code

Use text-based tools like Mermaid.js rather than image files. Diagrams-as-code can be:
- Versioned alongside the code in Git
- Reviewed by reading the text diff
- Used without a license

```mermaid
graph LR
  User[User] --> API
  API --> Retriever
  Retriever --> Vec[(Vector DB)]
  Retriever --> LLM
  LLM --> API
  API --> User
```

### Sequence and Data Flow Diagrams

Beyond the C4 model, two additional diagram types capture what static structure diagrams miss.

**Sequence diagrams** — capture timing and order of operations (who calls whom, in what order):

In [ ]:
render_diagram(Path("assets/17-sequence-diagram.png"))

**Data flow diagrams** — trace data and transformations as data moves through the system:

In [ ]:
render_diagram(Path("assets/18-data-flow-diagram.png"))

### Writing Architecture Decision Records (ADRs)

Diagrams show **what** — the structure, boxes, arrows, data flows. They don't capture **why** — the reasoning, constraints, and alternatives considered at the time.

Six months from now: "Why did we use pgvector instead of Pinecone?" Without an ADR, two outcomes: either the reason is forgotten and a silent regression is introduced (weeks to unwind), or the debate is repeated from scratch (weeks lost).

> An ADR is a short document that captures a single architectural decision. Not a massive corporate document — a quick, focused snapshot of an engineering choice.

#### ADR Guidelines

- Markdown files, numbered sequentially, stored in version control
- **Never delete an ADR** — only supersede it (old ADR links to new ADR)
- Superseded ADRs are historical record: they preserve why the team made a choice at the time, what assumptions were true back then, and how conditions evolved

#### The Standard ADR Format

```markdown
# ADR-XXXX: Title (short and descriptive)

**Status:** Proposed | Accepted | Deprecated | Superseded by ADR-YYYY
**Date:** YYYY-MM-DD

## Context
What prompted this decision?

## Decision
Stated clearly — what was decided.

## Alternatives Considered
What else was evaluated and why it was rejected.

## Consequences
Concrete positives and negatives.

## Trade-offs Accepted
What you explicitly gave up.

## Review Triggers
When you should reconsider this decision.
```

#### Connecting the Four-Step Framework to ADR Sections

| Framework step | ADR section |
|---|---|
| 1. Define constraints | Context |
| 2. Weighted matrix | Alternatives considered |
| 3. Trade-off assessment | Consequences + Trade-offs accepted |
| 4. Stakeholder alignment | Reviewers |

### Architecture Roadmaps, Technical Debt, and Stakeholder Communication

#### Architecture Evolution Roadmaps

A roadmap has three parts:
1. **Current state** — what's built today and its known limitations
2. **Target state** — where you'll be in 6–12 months, aligned with product and business
3. **Migration plan** — small, deployable increments, prioritized by impact × risk

#### Managing Technical Debt Systematically

Not all debt is bad — always categorize it:

| Type | Description |
|---|---|
| **Deliberate** | Strategic shortcut taken to ship fast |
| **Accidental** | Didn't realize it was suboptimal at the time |
| **Environmental** | The ecosystem changed beneath you |

**Prioritize by impact × effort:**

| Quadrant | Action |
|---|---|
| High impact + Low effort | Quick wins — do now |
| High impact + High effort | Big bets — plan and schedule |
| Low impact + Low effort | Fill-ins — batch as padding |
| Low impact + High effort | Avoid/defer |

**Tie debt to the roadmap, not a separate "tech debt sprint"** — if you're touching the prompt builder, pay down its debt at the same time.

Tracking tools:
- **Tickets** (Jira, GitHub Issues, Azure DevOps) — execution
- **Spreadsheets/CSV** — visibility
- **Markdown files** — version control
- A hybrid approach works best: use all three for their respective strengths

#### Stakeholder Communication

The same architectural change needs to be framed differently for each audience — what's compelling to engineering ("clean separation of concerns") means nothing to finance. Frame every change in the language of the audience who must approve or act on it.